# Biomarker Analysis Pipeline (v2)

1:1 line-matched ICI vs never-ICI cohorts with two propensity score models.

### Matching
- **1:1** — one control per ICI case, matched on (cancer_type, line_category)

### Propensity score models
- **embeddings_only** — LR on text embeddings
- **all_covariates** — LR on embeddings + demographics + cancer type + panel version + line

### Analysis tracks
- **Track 1a** — ICI-only, standard: `S(t) ~ base_vars + line_dummies + marker`
  - Weight variants: generalizability (1/ps), overlap (1−ps), unweighted
  - Tests sensitivity to population target definition
- **Track 1b** — ICI-only, prognostic-adjusted: `S(t) ~ base_vars + line_dummies + prog_score + marker`
  - CV-derived prognostic score from PCA-reduced clinical text embeddings + penalized CoxPH
  - Weight variants: unweighted (progAdj), generalizability-weighted (progAdj_weighted)
  - Tests whether signal survives adjustment for disease severity / unmeasured confounders
- **Track 2** — Full cohort, IPTW-weighted: `S(t) ~ base_vars + line_dummies + marker + ICI + marker×ICI`
  - Weight variants: ATE (stabilized), ATT, overlap (Li et al. JASA 2018), unweighted

### Robustness filtering
- **Track 1a**: markers significant in ≥2 weighting schemes with consistent HR direction
- **Track 1b**: markers significant in ≥2 prognostic-adjusted schemes with consistent HR direction
- **Confounding-robust**: markers passing BOTH Track 1a and 1b with consistent direction
- **Track 2**: markers significant in ≥2 schemes with consistent classifier

### Weighting approaches
- **ATE/ATT** — standard stabilized IPTW, truncated at (1st, 99th) percentiles
- **Overlap (OVL)** — w = 1−ps for treated, w = ps for control; targets equipoise population, bounded [0,1], no truncation needed
- **Generalizability** — ICI-only: w = 1/ps (standard) or w = 1−ps (overlap)

### Stages
1. **Data regeneration** — `generate_all_non_text_covariates.py` (SV/Fusion fix)
2. **Line-matched cohorts** — `build_line_matched_cohort.py`
3. **Propensity scores** — `ICI_LRs.py`
4. **IPTW datasets** — `generate_IPTW_df.py --ps_model {embeddings_only,all_covariates}` (includes clinical text embeddings for prognostic score)
5. **Cox models** — `run_IPTW_analysis.py --ps_model {embeddings_only,all_covariates}`
6. **Compile results** — cross-scheme robustness filtering across Tracks 1a, 1b, and 2

In [ ]:
import subprocess
import sys
import os

SCRIPT_DIR = os.path.dirname(os.path.abspath('__file__'))
PS_MODELS = ['embeddings_only', 'all_covariates']

def run_and_stream(label, cmd):
    """Run a command and stream its output inline."""
    print(f"\n--- {label} ---")
    result = subprocess.run(cmd, cwd=SCRIPT_DIR,
                            stdout=subprocess.PIPE, stderr=subprocess.PIPE,
                            universal_newlines=True)
    if result.stdout:
        print(result.stdout)
    if result.returncode != 0:
        print(f"FAILED (exit code {result.returncode})")
        if result.stderr:
            print(result.stderr)
        raise RuntimeError(f"{label} failed")
    print(f"Done: {label}")

## Stage 1: Data Regeneration (SV/Fusion fix)

Re-run `generate_all_non_text_covariates.py` to ensure `complete_somatic_data_df.csv` includes correct SV/Fusion columns.

In [ ]:
run_and_stream('Data regeneration',
               [sys.executable, '../data_preprocessing/generate_all_non_text_covariates.py'])

## Stage 2: Line-Matched Cohort Construction

Build 1:1 and 1:k matched cohorts on (cancer_type, line_category).

In [ ]:
run_and_stream('Line-matched cohorts',
               [sys.executable, 'build_line_matched_cohort.py'])

## Stage 3: Propensity Score Generation

Train embeddings-only and all-covariates LR propensity models for each matching scheme.

In [ ]:
run_and_stream('Propensity scores',
               [sys.executable, 'ICI_LRs.py'])

## Stage 4: IPTW Dataset Generation

Build IPTW datasets for each {matching, ps_model} combination.

In [ ]:
for ps_model in PS_MODELS:
    run_and_stream(f'IPTW dataset ({ps_model})',
                   [sys.executable, 'generate_IPTW_df.py', '--ps_model', ps_model])

## Stage 5: Cox Model Analysis

Run Track 1 (ICI-only, generalizability-weighted) and Track 2 (full-cohort interaction) for each specification.

In [ ]:
for ps_model in PS_MODELS:
    run_and_stream(f'Cox models ({ps_model})',
                   [sys.executable, 'run_IPTW_analysis.py', '--ps_model', ps_model])

## Stage 6: Compile Results

Aggregate significant hits across all specifications for comparison.

In [ ]:
import re
import numpy as np
import pandas as pd

OUTPUT_PATH = '/data/gusev/USERS/jpconnor/data/clinical_text_embedding_project/biomarker_analysis/'
COMPILED_PATH = os.path.join(OUTPUT_PATH, 'compiled_results/')
os.makedirs(COMPILED_PATH, exist_ok=True)

MATCHINGS = ['1to1']
MIN_SCHEMES_TRACK1A = 3  # out of 6 (3 weights x 2 PS models = 50%)
MIN_SCHEMES_TRACK1B = 3  # out of 4 (2 weights x 2 PS models = 75%)
MIN_SCHEMES_TRACK2 = 4   # out of 8 (4 weights x 2 PS models = 50%)

# Track 1a: standard weighting variants (no prognostic adjustment)
TRACK1A_WEIGHTS = ['weighted', 'OVL', 'unweighted']
# Track 1b: prognostic-score-adjusted variants
TRACK1B_WEIGHTS = ['progAdj', 'progAdj_weighted']
# Track 2: full-cohort interaction
TRACK2_WEIGHTS = ['ATE', 'ATT', 'OVL', 'noIPTW']

# Discover cancer types from result filenames
cancer_types = set()
for matching in MATCHINGS:
    for ps_model in PS_MODELS:
        spec = f'{matching}_{ps_model}'
        run_path = os.path.join(OUTPUT_PATH, f'IPTW_runs_{spec}/')
        if not os.path.isdir(run_path):
            continue
        for fname in os.listdir(run_path):
            m = re.match(r'(.+)_track[12]_', fname)
            if m:
                cancer_types.add(m.group(1))
cancer_types = sorted(cancer_types)
print(f"Discovered cancer types: {cancer_types}")

# ================================================
# 1. Compile all significant hits
# ================================================

def _collect_track1_hits(weight_list, label):
    """Collect significant Track 1 hits for a given set of weight types."""
    rows = []
    for matching in MATCHINGS:
        for ps_model in PS_MODELS:
            spec = f'{matching}_{ps_model}'
            run_path = os.path.join(OUTPUT_PATH, f'IPTW_runs_{spec}/')
            for ct in cancer_types:
                for weight_type in weight_list:
                    fname = os.path.join(run_path, f'{ct}_track1_{weight_type}_ICI_only.csv')
                    if not os.path.exists(fname):
                        continue
                    df = pd.read_csv(fname)
                    if 'significant_marker' in df.columns:
                        hits = df.loc[df['significant_marker']].copy()
                        hits['matching'] = matching
                        hits['ps_model'] = ps_model
                        hits['weight_type'] = weight_type
                        hits['cancer_type'] = ct
                        rows.append(hits)
    compiled = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()
    print(f"{label}: {len(compiled)} significant hits across all specs")
    return compiled

# --- Track 1a: standard (no prognostic adjustment) ---
t1a_compiled = _collect_track1_hits(TRACK1A_WEIGHTS, "Track 1a (standard)")
t1a_compiled.to_csv(os.path.join(COMPILED_PATH, 'track1a_all_significant_hits.csv'), index=False)

# --- Track 1b: prognostic-adjusted ---
t1b_compiled = _collect_track1_hits(TRACK1B_WEIGHTS, "Track 1b (prognostic-adjusted)")
t1b_compiled.to_csv(os.path.join(COMPILED_PATH, 'track1b_all_significant_hits.csv'), index=False)

# --- Track 2: compile interaction hits ---
track2_rows = []
for matching in MATCHINGS:
    for ps_model in PS_MODELS:
        spec = f'{matching}_{ps_model}'
        run_path = os.path.join(OUTPUT_PATH, f'IPTW_runs_{spec}/')
        for ct in cancer_types:
            for weight_type in TRACK2_WEIGHTS:
                fname = os.path.join(run_path, f'{ct}_track2_{weight_type}_interaction.csv')
                if not os.path.exists(fname):
                    continue
                df = pd.read_csv(fname)
                if 'significant_predictive' in df.columns:
                    hits = df.loc[df['significant_predictive']].copy()
                    hits['matching'] = matching
                    hits['ps_model'] = ps_model
                    hits['weight_type'] = weight_type
                    hits['cancer_type'] = ct
                    track2_rows.append(hits)

track2_compiled = pd.concat(track2_rows, ignore_index=True) if track2_rows else pd.DataFrame()
track2_compiled.to_csv(os.path.join(COMPILED_PATH, 'track2_all_significant_hits.csv'), index=False)
print(f"Track 2: {len(track2_compiled)} significant hits across all specs")

# ================================================
# 2. Cross-scheme robustness filtering
# ================================================

def cross_scheme_filter_track1(df, min_schemes=2):
    """Keep markers significant in >= min_schemes with consistent HR direction."""
    if df.empty:
        return df
    df = df.copy()
    df['scheme'] = df['matching'] + '|' + df['ps_model'] + '|' + df['weight_type']
    grouped = df.groupby(['marker', 'cancer_type'])

    robust = []
    for (marker, ct), grp in grouped:
        n_schemes = grp['scheme'].nunique()
        if n_schemes < min_schemes:
            continue
        all_risk = (grp['HR_marker'] > 1).all()
        all_prot = (grp['HR_marker'] < 1).all()
        if not (all_risk or all_prot):
            continue
        if 'extreme_hr_flag' in grp.columns and grp['extreme_hr_flag'].any():
            continue
        robust.append({
            'marker': marker,
            'cancer_type': ct,
            'n_schemes': n_schemes,
            'direction': 'risk' if all_risk else 'protective',
            'HR_median': grp['HR_marker'].median(),
            'HR_min': grp['HR_marker'].min(),
            'HR_max': grp['HR_marker'].max(),
            'FDR_min': grp['FDR_marker'].min(),
            'FDR_max': grp['FDR_marker'].max(),
            'mutation_type': grp['mutation_type'].iloc[0],
        })
    return pd.DataFrame(robust)


def cross_scheme_filter_track2(df, min_schemes=2):
    """Keep markers significant in >= min_schemes with consistent classifier."""
    if df.empty:
        return df
    df = df.copy()
    df['scheme'] = df['matching'] + '|' + df['ps_model'] + '|' + df['weight_type']
    grouped = df.groupby(['marker', 'cancer_type'])

    robust = []
    for (marker, ct), grp in grouped:
        n_schemes = grp['scheme'].nunique()
        if n_schemes < min_schemes:
            continue
        classifiers = grp['classifier'].unique()
        if len(classifiers) > 1:
            continue
        if 'extreme_hr_flag' in grp.columns and grp['extreme_hr_flag'].any():
            continue
        has_inf = (~np.isfinite(grp['HR_markerxICI'])).any()
        if has_inf:
            continue

        sig_ici_count = (grp.get('significant_in_ICI', pd.Series(dtype=bool)) == True).sum()

        row = {
            'marker': marker,
            'cancer_type': ct,
            'n_schemes': n_schemes,
            'classifier': classifiers[0],
            'FDR_min': grp['FDR_markerxICI'].min(),
            'FDR_max': grp['FDR_markerxICI'].max(),
            'HR_ICI_median': grp['HR_marker_ICI'].median(),
            'HR_nonICI_median': grp['HR_marker_nonICI'].median(),
            'sig_in_ICI_count': sig_ici_count,
            'mutation_type': grp['mutation_type'].iloc[0],
        }

        for ec in ['n_ICI_pos', 'events_ICI_pos', 'n_nonICI_pos', 'events_nonICI_pos']:
            if ec in grp.columns:
                row[ec + '_median'] = grp[ec].median()

        robust.append(row)
    return pd.DataFrame(robust)


# --- Track 1a: weighting robustness (standard) ---
t1a_robust = cross_scheme_filter_track1(t1a_compiled, min_schemes=MIN_SCHEMES_TRACK1A)

# --- Track 1b: weighting robustness (prognostic-adjusted) ---
t1b_robust = cross_scheme_filter_track1(t1b_compiled, min_schemes=MIN_SCHEMES_TRACK1B)

# --- Confounding-robust: markers in BOTH 1a and 1b with consistent direction ---
if not t1a_robust.empty and not t1b_robust.empty:
    merge_cols = ['marker', 'cancer_type']
    t1_combined = t1a_robust[merge_cols + ['direction', 'HR_median', 'HR_min', 'HR_max',
                                            'FDR_min', 'FDR_max', 'n_schemes', 'mutation_type']].merge(
        t1b_robust[merge_cols + ['direction', 'HR_median', 'HR_min', 'HR_max',
                                  'FDR_min', 'FDR_max', 'n_schemes']],
        on=merge_cols, suffixes=('_1a', '_1b'))
    t1_confounding_robust = t1_combined[
        t1_combined['direction_1a'] == t1_combined['direction_1b']
    ].copy()
    t1_confounding_robust.rename(columns={'direction_1a': 'direction'}, inplace=True)
    t1_confounding_robust.drop(columns=['direction_1b'], inplace=True)
else:
    t1_confounding_robust = pd.DataFrame()

# --- Track 2 ---
t2_robust = cross_scheme_filter_track2(track2_compiled, min_schemes=MIN_SCHEMES_TRACK2)

print(f"\nTrack 1a cross-scheme robust (>={MIN_SCHEMES_TRACK1A} of 6 schemes): {len(t1a_robust)}")
print(f"Track 1b cross-scheme robust (>={MIN_SCHEMES_TRACK1B} of 4 schemes): {len(t1b_robust)}")
print(f"Track 1 confounding-robust (in both 1a AND 1b, consistent direction): {len(t1_confounding_robust)}")
print(f"Track 2 cross-scheme robust (>={MIN_SCHEMES_TRACK2} of 8 schemes): {len(t2_robust)}")

t1a_robust.to_csv(os.path.join(COMPILED_PATH, 'track1a_cross_scheme_robust.csv'), index=False)
t1b_robust.to_csv(os.path.join(COMPILED_PATH, 'track1b_cross_scheme_robust.csv'), index=False)
t1_confounding_robust.to_csv(os.path.join(COMPILED_PATH, 'track1_confounding_robust.csv'), index=False)
t2_robust.to_csv(os.path.join(COMPILED_PATH, 'track2_cross_scheme_robust.csv'), index=False)

# ================================================
# 3. Diagnostics summary
# ================================================
diag_rows = []
for matching in MATCHINGS:
    for ps_model in PS_MODELS:
        spec = f'{matching}_{ps_model}'
        run_path = os.path.join(OUTPUT_PATH, f'IPTW_runs_{spec}/')
        if not os.path.isdir(run_path):
            continue
        for ct in cancer_types:
            diag_path = os.path.join(run_path, f'{ct}_diagnostics/')
            ess_file = os.path.join(diag_path, 'effective_sample_sizes.csv')
            if os.path.isfile(ess_file):
                ess = pd.read_csv(ess_file)
                ess['matching'] = matching
                ess['ps_model'] = ps_model
                diag_rows.append(ess)

if diag_rows:
    diag_df = pd.concat(diag_rows, ignore_index=True)
    diag_df.to_csv(os.path.join(COMPILED_PATH, 'scheme_diagnostics_summary.csv'), index=False)
    print(f"\nDiagnostics summary saved ({len(diag_df)} rows)")

print(f"\nAll outputs saved to {COMPILED_PATH}")